In [ ]:
# ============================================
# 03_feature_engineering.ipynb
# Project: Customer Churn Analysis
# Purpose: Prepare data for ML models
#          - Encode categorical variables
#          - Scale numeric features
#          - Split into train/test sets
#          - Save prepared dataset
# ============================================

# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
from dotenv import load_dotenv
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import os
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

# Visualisation style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.titleweight'] = 'bold'

# Load and clean data
load_dotenv()
engine = create_engine(
    f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)
df = pd.read_sql('SELECT * FROM telco', engine)

# Apply cleaning
df['totalcharges'] = pd.to_numeric(
    df['totalcharges'].str.strip(), errors='coerce'
).fillna(0)
df['churn'] = df['churn'].map({'Yes': 1, 'No': 0})
df.columns = df.columns.str.lower().str.strip()
df = df.drop('customerid', axis=1)

print(f"Data loaded — {df.shape[0]} rows, {df.shape[1]} columns")
print(f"\nCategorical columns to encode:")
print(df.select_dtypes(include='object').columns.tolist())

In [ ]:
# ============================================
# Cell 2: Drop Irrelevant Features
# Based on correlation analysis in notebook 02
# gender:       -0.009 — near zero, no predictive value
# phoneservice: +0.012 — near zero, no predictive value
# ============================================

cols_to_drop = ['gender', 'phoneservice']
df = df.drop(cols_to_drop, axis=1)

print(f"Dropped: {cols_to_drop}")
print(f"Remaining shape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"\nRemaining columns:")
print(df.columns.tolist())

In [ ]:
# ============================================
# Cell 3: One-Hot Encode Categorical Variables
# Why one-hot vs label encoding:
# Label encoding implies false order (A=0, B=1, C=2 implies C > B > A which is meaningless for categories)
# One-hot creates binary columns — no order implied
# ============================================

# --- Engineer avg_monthly_spend BEFORE creating X ---
# Must happen here — before X is created from df
# totalcharges and tenure correlation = 0.829
# avg_monthly_spend captures pure spend signal independent of tenure — resolves multicollinearity
df['avg_monthly_spend'] = df.apply(
    lambda x: x['totalcharges'] / x['tenure']
    if x['tenure'] > 0 else x['monthlycharges'],
    axis=1
)
df = df.drop('totalcharges', axis=1)

print(f"avg_monthly_spend engineered")
print(f"Range: ${df['avg_monthly_spend'].min():.2f} "
      f"to ${df['avg_monthly_spend'].max():.2f}")
print(f"Mean: ${df['avg_monthly_spend'].mean():.2f}")

# Separate target variable before encoding
X = df.drop('churn', axis=1)
y = df['churn']

# One-hot encode all categorical columns
# drop_first=True removes one column per category
# to avoid multicollinearity (dummy variable trap)
X_encoded = pd.get_dummies(X, drop_first=True)

print(f"Shape before encoding: {X.shape}")
print(f"Shape after encoding:  {X_encoded.shape}")
print(f"\nNew columns created by encoding:")
original_cols = X.select_dtypes(include='object').columns.tolist()
new_cols = [col for col in X_encoded.columns
            if any(col.startswith(orig) for orig in original_cols)]
print(new_cols)
print(f"\nTarget variable (y) distribution:")
print(y.value_counts())

In [ ]:
# ============================================
# Cell 4: Scale Numeric Features
# Why scaling is needed:
# tenure:        0-72
# monthlycharges: 18-119
# totalcharges:  0-8684
# Without scaling — totalcharges dominates the model simply because its numbers are bigger
# StandardScaler brings everything to mean=0, std=1
# ============================================

from sklearn.preprocessing import StandardScaler

# Identify numeric columns to scale
# seniorcitizen is binary (0/1) — no scaling needed
# all one-hot encoded columns are binary — no scaling needed
# totalcharges replaced by avg_monthly_spend
numeric_cols = ['tenure', 'monthlycharges', 'avg_monthly_spend']

# Initialise scaler
scaler = StandardScaler()

# Copy encoded DataFrame
X_scaled = X_encoded.copy()

# Fit and transform numeric columns only
X_scaled[numeric_cols] = scaler.fit_transform(
    X_scaled[numeric_cols]
)

# Verify
print("=== BEFORE SCALING ===")
print(X_encoded[numeric_cols].describe().round(2))

print("\n=== AFTER SCALING ===")
print(X_scaled[numeric_cols].describe().round(2))

print("\nScaling complete — mean should be ~0, std should be ~1")

# Save scaler for use in scoring
import pickle
with open('../data/processed/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("Scaler saved to data/processed/scaler.pkl")

In [ ]:
# ============================================
# Cell 5: Train/Test Split
# Purpose: Split data into training and test sets
# Training set (80%) — model learns from this
# Test set (20%)     — model evaluated on unseen data
# random_state=42    — ensures reproducible split same split every time we run
# stratify=y         — ensures same churn ratio in both train and test sets
# ============================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("=== TRAIN/TEST SPLIT COMPLETE ===")
print(f"\nTotal dataset:  {len(X_scaled):,} rows")
print(f"Training set:   {len(X_train):,} rows ({len(X_train)/len(X_scaled)*100:.0f}%)")
print(f"Test set:       {len(X_test):,} rows ({len(X_test)/len(X_scaled)*100:.0f}%)")

print(f"\n=== CHURN RATIO CHECK ===")
print(f"Overall churn rate:  {y.mean()*100:.1f}%")
print(f"Training churn rate: {y_train.mean()*100:.1f}%")
print(f"Test churn rate:     {y_test.mean()*100:.1f}%")
print("\nRatios should match — confirms stratify=y worked correctly")

print(f"\n=== FEATURE SUMMARY ===")
print(f"Total features: {X_train.shape[1]}")
print(f"Feature names:")
for i, col in enumerate(X_scaled.columns, 1):
    print(f"  {i:2}. {col}")

In [ ]:
# ============================================
# Cell 6: Save Prepared Dataset
# Purpose: Save train/test sets for use in notebook 04 without rerunning entire feature engineering pipeline
# ============================================

import os

# Create processed data folder if it doesn't exist
os.makedirs('../data/processed', exist_ok=True)

# Save all four sets
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

# Save feature names for reference
feature_names = X_scaled.columns.tolist()
pd.Series(feature_names).to_csv(
    '../data/processed/feature_names.csv', index=False
)

print("=== DATASETS SAVED ===")
print(f"X_train: {X_train.shape} → data/processed/X_train.csv")
print(f"X_test:  {X_test.shape}  → data/processed/X_test.csv")
print(f"y_train: {y_train.shape} → data/processed/y_train.csv")
print(f"y_test:  {y_test.shape}  → data/processed/y_test.csv")
print(f"\nFeature names saved → data/processed/feature_names.csv")
print(f"Total features: {len(feature_names)}")

print(f"\n=== FEATURE ENGINEERING SUMMARY ===")
print(f"Original columns:     18")
print(f"Dropped:               2  (gender, phoneservice)")
print(f"After encoding:       28  (one-hot expanded categoricals)")
print(f"Numeric cols scaled:   3  (tenure, monthlycharges, totalcharges)")
print(f"Training rows:     {len(X_train):,}")
print(f"Test rows:         {len(X_test):,}")
print(f"Churn rate preserved: {y_test.mean()*100:.1f}% (stratified split)")
print(f"\nData is ready for ML model training in notebook 04")